# RNAstructure
<!-- SPDX-License-Identifier: GPL-3.0-only -->

Adapted from `sinc-lab/lncRNA-folding`.
Modified by Jingwen Liu, 2026, for full-length viral RNA benchmarking.

In [ ]:
import os
import pandas as pd
import time
import tarfile
import subprocess
from pathlib import Path
from tqdm import tqdm

In [ ]:
method_name = "RNAstructure"
base = Path.cwd()

In [ ]:
install_dir = base.parent / 'tools'
os.makedirs(install_dir, exist_ok=True)
RNAstructure_archive = install_dir / 'RNAstructureLinuxTextInterfaces64bit.tgz'
RNAstructure_path = install_dir / 'RNAstructure'

if not RNAstructure_path.exists() or not (RNAstructure_path / 'exe' / 'Fold').exists():
    %cd {install_dir}
    if not RNAstructure_archive.exists():
        !wget -q http://rna.urmc.rochester.edu/Releases/current/RNAstructureLinuxTextInterfaces64bit.tgz
    !tar xfz RNAstructureLinuxTextInterfaces64bit.tgz
else:
    print('RNAstructure already installed at', RNAstructure_path)

In [ ]:
RNAstructure_path = base.parent / 'tools'/ 'RNAstructure'
fold_bin = RNAstructure_path / 'exe' / 'Fold'
data_tables = RNAstructure_path / 'data_tables'

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
def run_folding(name: str, seq: str):
    with open('tmp.fasta', 'w') as ofile:
        ofile.write(f'>{name}\n{seq}\n')
    env = os.environ.copy()
    env['DATAPATH'] = str(data_tables)
    cmd = [str(fold_bin), '--bracket', '--MFE', 'tmp.fasta', 'tmp.dot']
    subprocess.run(cmd, env=env)
    in_lines = open('tmp.dot','r').readlines()
    with open('clean_tmp.dot','w') as out_file:
        for line in in_lines:
            if line[0]==">":
                sline = line.split(' ')
                out_file.write(">" + sline[4])
            else:
                out_file.write(line)
    return 'clean_tmp.dot'

In [ ]:
out_fasta_name = '../prediction/RNAstructure'
if os.path.exists(out_fasta_name + '.fasta'):
    os.remove(out_fasta_name + '.fasta')

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}")
for i, vid in enumerate(virus_ids):
    start_time = time.time()
    seq = viruses.loc[vid]['sequence']
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)

    dot_file_name = run_folding(vid, seq)
    os.system('cat ' + dot_file_name + ' >> ' + out_fasta_name + '.fasta')
    print(f"{time.time() - start_time: .1f} s")
    